# 04 – Serve Recommendations

We have trained embeddings. Now we use them to generate recommendations.

**The two-phase serving model:**
1. **Retrieval** — quickly find ~500 candidate movies that are broadly relevant
2. **Ranking** — score each candidate precisely and return the top N

In production these are separate systems. The retrieval phase uses an
approximate nearest-neighbor index (FAISS, Pinecone) for speed.
The ranking phase can be a more expensive model (neural net, gradient boosting).

Here both phases use dot-product similarity on ALS embeddings — simple and effective.

In [ ]:
import sys
sys.path.insert(0, "../src")

from rec_sys.data import MovieLensLoader
from rec_sys.data.schema import MovieId, UserId
from rec_sys.model import ALSTrainer
from rec_sys.pipeline.batch import BatchJob
from rec_sys.pipeline.serving import Recommender
from rec_sys.storage.user_cache import InMemoryUserCache
from rec_sys.storage.vector_db import InMemoryVectorDB

# Run the full batch job to populate all stores.
loader = MovieLensLoader("../data/ml-1m")
trainer = ALSTrainer(factors=64, iterations=20)
vector_db = InMemoryVectorDB()
user_cache = InMemoryUserCache()

job = BatchJob(loader=loader, trainer=trainer, vector_db=vector_db, user_cache=user_cache)
job.run()

rec = Recommender(
    vector_db=vector_db,
    user_cache=user_cache,
    trainer=trainer,
    popularity_scores=job.popularity_scores,
)
print("System ready.")

## Personalized Recommendations

`recommend_for_user` runs the full retrieval + filter + rank pipeline:
1. Get user's embedding from the trained model
2. Query vector DB for the N·3 nearest movies
3. Remove movies the user has already rated
4. Return top N by similarity score

In [ ]:
def show_recs(movielens_uid: int, n: int = 10) -> None:
    uid = loader.remapped_user_id(movielens_uid)
    user_ratings = user_cache.get_user_ratings(uid) or []

    print(f"User {movielens_uid}  ({len(user_ratings)} ratings in history)")
    print(f"\nRecently rated (sample):")
    for r in sorted(user_ratings, key=lambda x: x.rating, reverse=True)[:5]:
        movie = loader.get_movie(r.movie_id)
        print(f"  ★{r.rating:.0f}  {movie.title}")

    recs = rec.recommend_for_user(uid, n=n)
    print(f"\nTop {n} recommendations:")
    for rank, (mid, score) in enumerate(recs, 1):
        movie = loader.get_movie(MovieId(int(mid)))
        print(f"  {rank:2d}. {movie.title:<50s}  ({score:.4f})")

show_recs(1)

In [ ]:
# Try a different user profile.
show_recs(500)

## Similar Items

`get_similar_items` finds movies most similar to a reference movie.
The logic: use the movie's embedding as the query, find nearest neighbors,
exclude the reference itself.

This is what "customers also bought" / "because you watched X" surfaces.

In [ ]:
def show_similar(title_query: str, n: int = 10) -> None:
    # Case-insensitive substring match on title.
    matches = [
        (mid, movie) for mid, movie in loader.movies.items()
        if title_query.lower() in movie.title.lower()
    ]
    if not matches:
        print(f"No movies found matching '{title_query}'")
        return

    ref_id, ref_movie = matches[0]
    similar = rec.get_similar_items(ref_id, n=n)

    print(f"Movies similar to: {ref_movie.title}\n")
    for rank, (sid, score) in enumerate(similar, 1):
        sm = loader.get_movie(MovieId(int(sid)))
        print(f"  {rank:2d}. {sm.title:<50s}  ({score:.4f})")

show_similar("Toy Story")

In [ ]:
show_similar("Terminator")

## Cold-Start: New Users

A brand-new user has no history. We fall back to popular items.

The three weighting strategies show different tradeoffs:
- **count**: most-watched — broadly safe but obvious
- **average**: highest-rated — biased toward obscure films with few reviews
- **hybrid**: count × average — balances breadth and quality

In [ ]:
from rec_sys.model.cold_start import PopularityFallback

for weighting in ("count", "average", "hybrid"):
    popularity = PopularityFallback.compute_popularity(loader.ratings, weighting=weighting)
    top5 = PopularityFallback.top_k_popular(popularity, k=5)
    print(f"Top 5 by '{weighting}':")
    for rank, (mid, score) in enumerate(top5, 1):
        movie = loader.get_movie(MovieId(int(mid)))
        print(f"  {rank}. {movie.title:<50s}  ({score:.1f})")
    print()

## The Full Pipeline at a Glance

```
BATCH (offline, runs daily)
─────────────────────────────────────────
MovieLensLoader  →  ratings matrix
ALSTrainer       →  user vectors + item vectors
InMemoryVectorDB ←  item vectors stored
InMemoryUserCache←  user history stored
PopularityFallback←  cold-start scores stored

SERVING (online, per request)
─────────────────────────────────────────
UserId → get user vector from trainer
         → query_nearest(user_vec, k=500) from VectorDB
         → filter already-seen items from UserCache
         → top_k(n=10) by dot product
         → return recommendations
```

For new users: skip to popularity fallback.

## What to Try Next

- Swap `InMemoryVectorDB` for **FAISS** (approximate nearest neighbors, 100x faster at scale)
- Add a **re-ranking** step with additional signals (recency, diversity, novelty)
- Implement **temporal decay**: downweight old ratings to capture taste drift
- Explore **hybrid models**: blend collaborative filtering with item content features (genres, directors)
- Add an **A/B test framework**: compare two recommendation strategies with real users